# Gray-GOOSE: A Public Intrusion Dataset and Comprehensive Analysis of Grayhole Attacks on the IEC 61850 GOOSE Protocol - Base Experiments

This notebooks runs the complete pipeline of the train, evaluate and explainability for **two experiencial scenarios**:

<!-- This notebook execute the complete training, evaluation and explainabality pipelina to **two experimentation scenarios**: -->

| Scenario | Name | Description |
|---------|------|-----------|
| `(1) Enriched Delta Features Scenario` | With delta features | Use **all** of the features, incluiding delta's |
| `(2) Original Atributes Scenario` | Without delta features | **Remove** the delta features, using only original attributes and attributes related to the SV protocol. |

---

### How to use this notebook

1. **Configure the active scenario** in the *"Active Scenario Selection"* cell;
2. **Execute all of the cells** in order (`Kernel > Restart & Run All`);
3. To run the **other scenario**, return at selection cell, change the value and rerun;
4. Alternatively, use the **"Execute all of the Scenarios"** cell in the final to run both the scenarios at once.

> **Warning:** The complete pipeline (with SHAP) can be late. The SHAP analysis is disabled by default — enable in the corresponding section when necessary.

---
## 📦 1. Imports

In [1]:
import os
import sys
import gc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set the project path and ensure we're in the correct directory
PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)

# Pipeline modules
from data.loader import load_dataset
from data.preprocess import preprocess
from model.train import train_model
from model.evaluate import evaluate_models, save_metrics_report
from explainability.shap_analysis import run_shap

print('✓ Successful imports included!')

✓ Successful imports included!


---
## ⚙️ 2. Scenarios Definition

All parameters for both scenarios are defined here, in a single place.
The values described are the paper's default values.

In [2]:
# ============================================================
# SHARED PARAMETERS (same in both scenarios)
# ============================================================

RANDOM_STATE = 42
N_SPLITS     = 5
DATASET_PATH = "./data/CSV files/gray-GOOSE.csv"
CLASS_NAMES  = ["SAG.DB", "FRG", "SAG.PB", "SAG.PBM", "Normal"]
MODEL_TYPE   = "xgboost"

# Columns always descarted - defined in the paper
BASE_DISCARTED_COLUMNS = [
    'ethDst', 'ethSrc', 'gocbRef', 'datSet', 'goID', 'test',
    'ndsCom', 'protocol', 'ethType', 'TPID', 'gooseAppid', 'class'
]

# Delta Features — Removed only in Scenario 2
DELTA_FEATURES = [
    'stDiff', 'sqDiff', 'gooseLengthDiff', 'cbStatusDiff', 'apduSizeDiff',
    'frameLengthDiff', 'timestampDiff', 'tDiff', 'timeFromLastChange',
]

# XGBoost parameters - the outhers parameters are set to default values - defined in the paper
XGBOOST_PARAMS = {
    "objective"    : 'multi:softprob',
    "eval_metric"  : 'mlogloss',
    "random_state" : RANDOM_STATE,
}

# Graphics to be generated in both scenarios
GRAPHICS = [
    "Bar Plot",
    "Beeswarm Summary Plot",
]

# ============================================================
# SCENARIOS DEFINITION
# ============================================================

SCENARIOS = {

    # ----------------------------------------------------------
    # SCENARIO 1 — With enriched delta features
    # Use all of the available features, incluiding the calculated
    # of diferences between consecutive packages.
    # ----------------------------------------------------------
    "(1) Enriched Delta Features Scenario": {
        "description"              : "All of the features, including delta",
        "without_delta_feats": False,
        "discarted_columns"        : BASE_DISCARTED_COLUMNS,
        "path_base"                : "./results_with_delta_features",
        "model_type"               : MODEL_TYPE,
        "model_params"             : XGBOOST_PARAMS,
        "n_splits"                 : N_SPLITS,
        "random_state"             : RANDOM_STATE,
        "graphics"                 : GRAPHICS,
        "class_names"              : CLASS_NAMES,
        "dataset_path"             : DATASET_PATH,
    },

    # ----------------------------------------------------------
    # SCENARIO 2 — Without delta features
    # Remove the delta features, using only the original 
    # atributes of GOOSE protocol and SV enriched and original attributes.
    # ----------------------------------------------------------
    "(2) Original Attributes Scenario": {
        "description"              : "Only original attributes - GOOSE attributes and SV enriched and original attributes",
        "without_delta_feats": True,
        "discarted_columns"        : BASE_DISCARTED_COLUMNS + DELTA_FEATURES,
        "path_base"                : "./results_without_delta_features",
        "model_type"               : MODEL_TYPE,
        "model_params"             : XGBOOST_PARAMS,
        "n_splits"                 : N_SPLITS,
        "random_state"             : RANDOM_STATE,
        "graphics"                 : GRAPHICS,
        "class_names"              : CLASS_NAMES,
        "dataset_path"             : DATASET_PATH,
    },
}

print(f'✓ {len(SCENARIOS)} scenarios defined:')
for name, cfg in SCENARIOS.items():
    print(f'   • {name}')
    print(f'     └─ {cfg["description"]}')

✓ 2 scenarios defined:
   • (1) Enriched Delta Features Scenario
     └─ All of the features, including delta
   • (2) Original Attributes Scenario
     └─ Only original attributes - GOOSE attributes and SV enriched and original attributes


---
## 🎯 3. Active Scenario Selection

**Modify only this cell** to change the scenario that will be executed in follow sections.

In [3]:
# ============================================================
# CHANGE THE VALUE BELOW TO SWITCH BETWEEN SCENARIOS
# ============================================================

ACTIVE_SCENARIO = "(1) Enriched Delta Features Scenario"
# ACTIVE_SCENARIO = "(2) Original Attributes Scenario"

# ============================================================

cfg = SCENARIOS[ACTIVE_SCENARIO]

print('=' * 60)
print(f'🚀 ACTIVED SCENARIO: {ACTIVE_SCENARIO}')
print('=' * 60)
print(f'  Description         : {cfg["description"]}')
print(f'  Dataset           : {cfg["dataset_path"]}')
print(f'  Results in     : {cfg["path_base"]}')
print(f'  Model            : {cfg["model_type"].upper()}')
print(f'  K-Fold splits     : {cfg["n_splits"]}')
print(f'  Random state      : {cfg["random_state"]}')
print(f'  Delta features    : {"Removed" if cfg["without_delta_feats"] else "Included"}')
print(f'  Descarted Columns ({len(cfg["discarted_columns"])}): {cfg["discarted_columns"]}')

🚀 ACTIVED SCENARIO: (1) Enriched Delta Features Scenario
  Description         : All of the features, including delta
  Dataset           : ./data/CSV files/gray-GOOSE.csv
  Results in     : ./results_with_delta_features
  Model            : XGBOOST
  K-Fold splits     : 5
  Random state      : 42
  Delta features    : Included
  Descarted Columns (12): ['ethDst', 'ethSrc', 'gocbRef', 'datSet', 'goID', 'test', 'ndsCom', 'protocol', 'ethType', 'TPID', 'gooseAppid', 'class']


---
## 📂 4. Data Loader

In [4]:
print('=' * 60)
print('STEP 1: Data Loader')
print('=' * 60)

if not os.path.exists(cfg['dataset_path']):
    raise FileNotFoundError(f"Dataset not found: {cfg['dataset_path']}")

print(f'dataset loading: {cfg["dataset_path"]}')
df = load_dataset(
    cfg['dataset_path']
)

print(f'✓ Dataset load: {df.shape[0]:,} lines × {df.shape[1]} columns')
print('First lines of the dataset:')
df.head()

STEP 1: Data Loader
dataset loading: ./data/CSV files/gray-GOOSE.csv
✓ Dataset load: 1,006,989 lines × 52 columns
First lines of the dataset:


,Time,isbA,isbB,isbC,vsbA,vsbB,vsbC,isbARmsValue,isbBRmsValue,isbCRmsValue,...,sqDiff,gooseLengthDiff,cbStatusDiff,apduSizeDiff,frameLengthDiff,timestampDiff,tDiff,timeFromLastChange,delay,class
0,1964138.9,0.163272,-0.073301,-0.066864,0.011570,-0.089145,0.077085,0.599404,0.504646,0.139568,...,1,0,0,0,0,1.010631,0.0,41.320489,0.051799,normal
1,1426260.5,-0.056559,-0.013255,0.067457,-0.009555,0.087962,-0.078398,0.599404,0.504646,0.139568,...,1,0,0,0,0,0.632000,0.0,0.832000,-0.061690,normal
2,5952282.5,-0.038440,0.074184,-0.035297,0.094951,-0.040145,-0.054785,0.106195,0.107485,0.108444,...,1,0,0,0,0,1.019966,0.0,2.861785,-0.031905,normal
3,1734536.6,-1.325588,-0.111646,0.698040,-0.005292,-0.078945,0.089856,0.113184,0.109433,0.079809,...,1,0,0,0,0,1.008903,0.0,14.031980,0.013290,normal
4,1042714.5,-0.070504,0.042741,0.026357,0.059772,0.035805,-0.095555,0.106195,0.107485,0.108444,...,1,0,0,0,0,1.011711,0.0,2.866502,-0.027188,normal


---
## 🔧 5. Pre-processing

In [5]:
print('=' * 60)
print('STEP 2: Pré-processing')
print('=' * 60)

X, y, encoder = preprocess(
    df,
    target_column='class',
    discarted_columns=cfg['discarted_columns']
)

print(f'✓ Features   : {X.shape[1]} colunas')
print(f'✓ Samples   : {len(y):,}')
print(f'✓ Classes    : {cfg["class_names"]}')
print()
print('Classes Distribution:')
for i, cls in enumerate(cfg['class_names']):
    count = (y == i).sum()
    pct = (count / len(y)) * 100
    bar = '█' * int(pct / 2)
    print(f'  {cls:12} : {count:6,} samples ({pct:5.1f}%)  {bar}')

# Clean up the original DataFrame to free memory
del df
gc.collect()
print()
print('✓ Pre-processing completed!')

STEP 2: Pré-processing
✓ Features   : 40 colunas
✓ Samples   : 1,006,989
✓ Classes    : ['SAG.DB', 'FRG', 'SAG.PB', 'SAG.PBM', 'Normal']

Classes Distribution:
  SAG.DB       : 102,470 samples ( 10.2%)  █████
  FRG          : 102,236 samples ( 10.2%)  █████
  SAG.PB       : 100,373 samples ( 10.0%)  ████
  SAG.PBM      : 101,910 samples ( 10.1%)  █████
  Normal       : 600,000 samples ( 59.6%)  █████████████████████████████

✓ Pre-processing completed!


---
## 🏋️ 6. Training with K-Fold Validation

In [6]:
cv_models, final_model = train_model(
    X, y,
    model_type=cfg['model_type'],
    params=cfg['model_params'],
    n_splits=cfg['n_splits'],
    seed=cfg['random_state']
)

STEP: K-Fold Cross-Validation (5 folds)

Training fold 1/5...
  - Training: 805591 samples
  - Validation: 201398 samples
  - Training model...

Training fold 2/5...
  - Training: 805591 samples
  - Validation: 201398 samples
  - Training model...

Training fold 3/5...
  - Training: 805591 samples
  - Validation: 201398 samples
  - Training model...

Training fold 4/5...
  - Training: 805591 samples
  - Validation: 201398 samples
  - Training model...

Training fold 5/5...
  - Training: 805592 samples
  - Validation: 201397 samples
  - Training model...

✓ Cross-Validation completed!

STEP 3: Final model training with all data (to shap values and production)
Training final model with all 1006989 samples...
✓ Final model trained!



---
## 📊 7. Evaluation

In [7]:
cv_metrics, kappa_mean, kappa_ci, cv_total_cm, all_y_true, all_y_pred = evaluate_models(
    cv_models, cfg['class_names']
)

EVALUATION: Metrics from Cross-Validation
Fold 1: Kappa = 0.9058
Fold 2: Kappa = 0.9059
Fold 3: Kappa = 0.9050
Fold 4: Kappa = 0.9048
Fold 5: Kappa = 0.9062

✓ Cohen's Kappa (CV): 0.9055 ± 0.0007
✓ Predictions collected for McNemar: 1006989 samples

Aggregated Confusion Matrix (CV - sum of all folds):
[[102449      2     15      4      0]
 [    56  90984    295   9628   1273]
 [   389     41  98377   1368    198]
 [    57   7978   1771  91998    106]
 [    25   7754   1229  26874 564118]]



---
## 📋 8. Final Results

In [8]:
print('=' * 60)
print(f'RESULTS — {ACTIVE_SCENARIO}')
print('=' * 60)

# Table of metrics by class
results_rows = []
for i, cls in enumerate(cfg['class_names']):
    results_rows.append({
        'Classe'     : cls,
        'F1-Score'   : f"{cv_metrics['F1-score Mean'][i]:.4f} ± {cv_metrics['F1-score CI'][i]:.4f}",
        'Precision'  : f"{cv_metrics['Precision Mean'][i]:.4f} ± {cv_metrics['Precision CI'][i]:.4f}",
        'Recall'     : f"{cv_metrics['Recall Mean'][i]:.4f} ± {cv_metrics['Recall CI'][i]:.4f}",
    })

results_df = pd.DataFrame(results_rows).set_index('Classe')
display(results_df)

print()
print(f" Accuracy Global (CV) : {cv_metrics['Global Accuracy Mean']:.4f} ± {cv_metrics['Global Accuracy CI']:.4f}")
print(f"Cohen's Kappa   (CV) : {kappa_mean:.4f} ± {kappa_ci:.4f}")

RESULTS — (1) Enriched Delta Features Scenario


,F1-Score,Precision,Recall
Classe,,,
SAG.DB,0.9973 ± 0.0003,0.9949 ± 0.0005,0.9998 ± 0.0001
FRG,0.8707 ± 0.0013,0.8524 ± 0.0125,0.8899 ± 0.0109
SAG.PB,0.9737 ± 0.0011,0.9675 ± 0.0019,0.9801 ± 0.0020
SAG.PBM,0.7938 ± 0.0018,0.7084 ± 0.0049,0.9027 ± 0.0092
Normal,0.9679 ± 0.0006,0.9972 ± 0.0004,0.9402 ± 0.0011



 Accuracy Global (CV) : 0.9413 ± 0.0005
Cohen's Kappa   (CV) : 0.9055 ± 0.0007


---
## 💾 9. Save Metrics Report

In [9]:
dataset_name = os.path.basename(cfg['dataset_path']).replace('.csv', '').replace('.parquet', '')

md_path, log_path = save_metrics_report(
    cv_metrics, kappa_mean, kappa_ci,
    cfg['class_names'],
    dataset_name,
    output_dir=cfg['path_base'],
    cv_total_cm=cv_total_cm
)

print(f'✓ Markdown Report: {md_path}')
print(f'✓ Log Report: {log_path}')

Saved confusion matrix SVG: ./results_with_delta_features\confusion_matrix_gray-GOOSE_20260327_154258.svg
✓ Markdown Report: ./results_with_delta_features\metrics_report_gray-GOOSE_20260327_154258.md
✓ Log Report: ./results_with_delta_features\metrics_report_gray-GOOSE_20260327_154258.log


---
## 🔍 10. Explainability Analysis (SHAP)

> The SHAP analysis is **disabled by default** because it can be computationally intensive.  
> To able, change `RUN_SHAP = False` to `RUN_SHAP = True` in the cell below.
>
> Since the dataset is **multiclass** (5-class), the SHAP will return values in 3D.  
> `(n_samples × n_features × n_classes)` and it will generate individual graphs for each class.

In [ ]:
# ============================================================
# CHANGE TO True TO EXECUTE SHAP
# ============================================================
RUN_SHAP = False
# ============================================================

if RUN_SHAP:
    print(f"Generation of SHAP graphics completed! See the results: {cfg['path_base']}")
    # Use the last fold of validation as the input set for SHAP 
    if cv_models and len(cv_models) > 0:
        _, shap_X, _ = cv_models[-1]
        print(f'Using last validation fold for SHAP: {len(shap_X):,} samples')
    else:
        shap_X = X
        print('Warning: cv_models is empty — using entire X set as fallback')

    run_shap(
        final_model,
        shap_X,
        cfg['class_names'],
        dataset_name=dataset_name,
        path_base=cfg['path_base'],
        graphics=cfg['graphics']
    )
    
    print(f"Generation of SHAP graphics completed! See the results: {cfg['path_base']}")
else:
    print("SHAP execution skipped. Set RUN_SHAP = True to enable it.")

Generation of SHAP graphics completed! See the results: ./results_with_delta_features
Using last validation fold for SHAP: 201,397 samples
EXPLAINABILITY: SHAP Analysis
✓ Using test set (201397 samples) for SHAP.

Creating SHAP explainer...
Calculating SHAP values for 201397 samples...
(this might take a while...)
✓ SHAP values calculated!

Calculating the 10 most important features based on mean absolute SHAP values...
               feature  importance  percentage
30              stDiff    1.689748   22.671303
22               StNum    1.307530   17.543093
36       timestampDiff    0.830116   11.137650
19                   t    0.794320   10.657367
38  timeFromLastChange    0.765101   10.265346
0                 Time    0.488252    6.550866
8         isbBRmsValue    0.448910    6.023013
21               SqNum    0.412130    5.529541
17     vsbBTrapAreaSum    0.374625    5.026340
14     isbBTrapAreaSum    0.342513    4.595483
✓ Multiclass classification detected (SHAP in 3D)

Generati

---
## 🔄 11. Execute all of the Scenarios (automatic loop)

This section execute **both of scenarios in sequence**, without necessity to change anything manually.
Util to run the complete experiment at once (`Kernel > Restart & Run All`).

> Execute this section only if wanted run the both of scenarios at once.  
> If you already execute the 4-10 sections to one especific scenario, it's not necessary to run it.

In [ ]:
# ============================================================
# Change these variables to control the execution of multiple scenarios in loop
# ============================================================
RUN_ALL_SCENARIOS = True
RUN_SHAP_IN_LOOP  = True   # able also SHAP in the loop, if desired
# ============================================================

if RUN_ALL_SCENARIOS:
    all_results = {}

    for scenario_name, scenario_cfg in SCENARIOS.items():
        print()
        print('█' * 60)
        print(f'  STARTING: {scenario_name}')
        print(f'  {scenario_cfg["description"]}')
        print('█' * 60)

        # --- Carregamento ---
        df_loop = load_dataset(
            scenario_cfg['dataset_path']
        )

        # --- Pre-processing ---
        X_loop, y_loop, encoder_loop = preprocess(
            df_loop,
            target_column='class',
            discarted_columns=scenario_cfg['discarted_columns']
        )
        del df_loop
        gc.collect()

        # --- Training ---
        cv_loop, final_loop = train_model(
            X_loop, y_loop,
            model_type=scenario_cfg['model_type'],
            params=scenario_cfg['model_params'],
            n_splits=scenario_cfg['n_splits'],
            seed=scenario_cfg['random_state']
        )

        # --- Evaluation ---
        cv_m, kappa_m, kappa_c, cm_loop, all_y_true, all_y_pred = evaluate_models(
            cv_loop, scenario_cfg['class_names']
        )

        # --- Save Report ---
        ds_name = os.path.basename(scenario_cfg['dataset_path']).replace('.csv','').replace('.parquet','')
        save_metrics_report(
            cv_m, kappa_m, kappa_c,
            scenario_cfg['class_names'],
            ds_name,
            output_dir=scenario_cfg['path_base'],
            cv_total_cm=cm_loop
        )

        # --- SHAP ---
        if RUN_SHAP_IN_LOOP:
            _, shap_X_loop, _ = cv_loop[-1]
            run_shap(
                final_loop,
                shap_X_loop,
                scenario_cfg['class_names'],
                dataset_name=ds_name,
                path_base=scenario_cfg['path_base'],
                graphics=scenario_cfg['graphics']
            )

        # --- Put a resume to final comparison ---
        all_results[scenario_name] = {
            'Accuracy'     : f"{cv_m['Global Accuracy Mean']:.4f} ± {cv_m['Global Accuracy CI']:.4f}",
            "Cohen's Kappa": f"{kappa_m:.4f} ± {kappa_c:.4f}",
            'Output Dir'   : scenario_cfg['path_base'],
        }

        # Clean memory between scenarios
        del X_loop, y_loop, cv_loop, final_loop
        gc.collect()

        print(f'\n✅ Scenario "{scenario_name}" completed!\n')

    # --- final comparative table ---
    print()
    print('=' * 60)
    print('SCENARIOS COMPARASION')
    print('=' * 60)
    comparison_df = pd.DataFrame(all_results).T
    display(comparison_df)

else:
    print('⏭️  Scenario loop desactived. Change RUN_ALL_SCENARIOS = True to execute.')


████████████████████████████████████████████████████████████
  STARTING: (1) Enriched Delta Features Scenario
  All of the features, including delta
████████████████████████████████████████████████████████████
STEP: K-Fold Cross-Validation (5 folds)

Training fold 1/5...
  - Training: 805591 samples
  - Validation: 201398 samples
  - Training model...

Training fold 2/5...
  - Training: 805591 samples
  - Validation: 201398 samples
  - Training model...

Training fold 3/5...
  - Training: 805591 samples
  - Validation: 201398 samples
  - Training model...

Training fold 4/5...
  - Training: 805591 samples
  - Validation: 201398 samples
  - Training model...

Training fold 5/5...
  - Training: 805592 samples
  - Validation: 201397 samples
  - Training model...

✓ Cross-Validation completed!

STEP 3: Final model training with all data (to shap values and production)
Training final model with all 1006989 samples...
✓ Final model trained!

EVALUATION: Metrics from Cross-Validation
Fold 1:

,Accuracy,Cohen's Kappa,Output Dir
(1) Enriched Delta Features Scenario,0.9413 ± 0.0005,0.9055 ± 0.0007,./results_with_delta_features
(2) Original Attributes Scenario,0.8168 ± 0.0007,0.7231 ± 0.0010,./results_without_delta_features


---

## ✅ Pipeline completed

The results was saved in configured directories for each scenario:

| Scenario | Output Directory |
|---------|-------------------|
| `(1) Enriched Delta Features Scenario` | `./results_with_delta_features/` |
| `(2) Original Atributes Scenario` | `./results_without_delta_features/` |

Each directory will contain:
- `metrics_report_*.md` — Report formated in Markdown
- `metrics_report_*.log` — Report in pure text
- `confusion_matrix_*.svg` — Visual confusion matrix
- *(if SHAP disable)* subdirectories `Bar Plot/` and `Beeswarm Summary Plot/` with the graphs for each class